# Method 2 — Cross-Encoder (Kaggle T4)

Phase 3 của `docs/method2_plan.md`: sinh nhãn, curriculum 2 giai đoạn, gate theo head. Ngân sách ~4.5h GPU.

**Ba quy tắc sống còn trên Kaggle** (§8 `docs/method2_plan.md`):

1. Bật **Save & Run All (Commit)** cho job dài — session tương tác bị ngắt sau ~20 phút không tương tác, commit run chạy nền đủ 12h.
2. Checkpoint mỗi 500 step vào `/kaggle/working`, và **luôn** hỗ trợ `resume_from`.
3. Cache model HuggingFace thành Kaggle Dataset (`BAAI/bge-m3` ~2.3GB) thay vì tải lại mỗi session.


In [ ]:
# ===== Cell 0: HF cache — PHẢI đặt trước mọi import transformers =====
# Thư viện đọc HF_HOME lúc import; set sau thì nó đã chốt cache mặc định.
import os
from pathlib import Path

# Tuỳ cách upload, dataset có thể lồng thêm một tầng. Dò cả hai thay vì
# hardcode: HF_HOME phải trỏ vào thư mục CHỨA `hub/`.
CANDIDATES = [
    Path('/kaggle/input/toolcalling-vi-hf-cache'),
    Path('/kaggle/input/toolcalling-vi-hf-cache/toolcalling-vi-hf-cache'),
    Path('/kaggle/input/toolcalling-vi-hf-cache/hf-cache'),
]
HF_HOME = next((p for p in CANDIDATES if (p / 'hub').is_dir()), None)
assert HF_HOME is not None, (
    'Không tìm thấy thư mục chứa hub/. Đã thử: '
    + ', '.join(str(p) for p in CANDIDATES)
)

os.environ['HF_HOME'] = str(HF_HOME)
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'

print('HF_HOME:', HF_HOME)
cached = sorted(p.name for p in (HF_HOME / 'hub').glob('models--*'))
for name in cached:
    print(' -', name)

# Thiếu trọng số thì lỗi chỉ lộ ra lúc nạp model, sau khi đã tốn thời gian.
for name in ('models--BAAI--bge-m3', 'models--xlm-roberta-base'):
    assert name in cached, f'Cache thiếu {name}'
    big = [
        f for f in (HF_HOME / 'hub' / name).rglob('*')
        if f.is_file() and f.suffix in ('.safetensors', '.bin') and f.stat().st_size > 10**8
    ]
    assert big, f'{name}: không có file trọng số > 100 MB'
print('\nHF cache OK')


In [ ]:
# ===== Cell 1: env — PIN version =====
# Ba package này quyết định API training VÀ tên metric của
# InformationRetrievalEvaluator. Đổi bản là đổi khoá metric, hỏng cả
# load_best_model_at_end lẫn khả năng so sánh giữa các run.
!pip install -q 'transformers==5.15.1' 'sentence-transformers==6.0.0' 'peft==0.20.0' \
                accelerate jsonschema rank_bm25 datasets

# torch KHÔNG pin: Kaggle cài sẵn bản CUDA riêng, ép cài lại vừa chậm vừa
# dễ lệch CUDA runtime của image. Chỉ ghi nhận version vào manifest.
import torch

free, total = torch.cuda.mem_get_info()
print(torch.cuda.get_device_name(0), f'{free/1024**3:.1f} / {total/1024**3:.1f} GB free')
print('bf16 supported:', torch.cuda.is_bf16_supported())
# Kỳ vọng: Tesla T4, ~15.0 GB free, bf16 = False → mọi config dùng fp16.


In [ ]:
# ===== Cell 2: mount code + data =====
# Dựng 3 dataset bằng: python scripts/method2/build_kaggle_upload.py --hf-cache
# Dataset mount ở /kaggle/input/<tên>/ với đúng cấu trúc script đã tạo.
!cp -r /kaggle/input/toolcalling-vi-src/src /kaggle/working/
!cp -r /kaggle/input/toolcalling-vi-src/configs /kaggle/working/

# toolcalling-vi-data để method2/ custom_vi/ benchmark_vi/ ở GỐC dataset,
# còn code lại tham chiếu đường dẫn tương đối `data/method2/...`.
!mkdir -p /kaggle/working/data
!cp -r /kaggle/input/toolcalling-vi-data/* /kaggle/working/data/
%cd /kaggle/working

import json, glob, sys
sys.path.insert(0, '/kaggle/working')
# HF_HOME đã set ở Cell 0 và được kế thừa sang mọi tiến trình con `!python`.

manifest = json.load(open('data/method2/manifest.json', encoding='utf-8'))
print('snapshot commit:', manifest.get('git_commit'))


## Sinh cặp (query, parameter) và kiểm tra tỉ lệ SKIP

**Gate §1.5**: nếu %SKIP > 30% thì dừng lại, xem lại quyết định Q2 (có thêm
fuzzy span alignment hay không) trước khi train.


In [ ]:
!python -m src.models.crossencoder.dataset --config configs/method2/crossencoder.yaml

stats = json.load(open('data/method2/label_stats.json', encoding='utf-8'))
print('SKIP rate:', stats['skip_rate'])
print('theo lý do:', stats['skip_by_reason'])
print('unsupported coverage:', stats['unsupported_type_coverage'])
assert stats['skip_rate'] <= 0.30, 'SKIP quá ngưỡng — xem lại Q2 trước khi train'


## Train — curriculum 2 giai đoạn, resume-safe

1. Warm-up trên glaive+xLAM (2 epoch) — học kỹ năng span tổng quát.
2. Fine-tune trên custom_vi (2 epoch, lr 1e-5) — domain đích.


In [ ]:
RUN = '/kaggle/working/artifacts/method2/crossencoder/run01'
resume = sorted(glob.glob(f'{RUN}/checkpoint-*'))[-1] if glob.glob(f'{RUN}/checkpoint-*') else None
print('resume from:', resume)

!python -m src.models.crossencoder.train \
    --config configs/method2/crossencoder.yaml \
    --output-dir {RUN} \
    --resume-from {resume}


## Gate để qua Phase 4 (chế độ oracle retrieval)

`has_value` F1 ≥ 0.90 · Span EM ≥ 0.80 · Enum acc ≥ 0.90 · Boolean acc ≥ 0.85


In [ ]:
!python -m src.models.crossencoder.evaluate \
    --config configs/method2/crossencoder.yaml \
    --model {RUN}/final \
    --pairs data/method2/crossencoder/val.jsonl \
    --output results/method2/metrics/crossencoder_val.json


In [ ]:
# ===== Lưu artifact =====
# Kaggle chỉ giữ /kaggle/working (20GB). Nén để tải về hoặc làm Dataset mới.
!tar czf /kaggle/working/crossencoder_run.tar.gz -C /kaggle/working/artifacts/method2 .
!du -h /kaggle/working/crossencoder_run.tar.gz
